In this notebook, I will use the data generated in poseRecognition.ipynb to extract additional features. The primary focus will be on calculating the angles between joints and measuring the speed of movement.

In [19]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [20]:
import pandas as pd


In [21]:
path = "output\pose_13.csv"

<>:1: SyntaxWarning: invalid escape sequence '\p'
<>:1: SyntaxWarning: invalid escape sequence '\p'
C:\Users\David\AppData\Local\Temp\ipykernel_25420\1398903690.py:1: SyntaxWarning: invalid escape sequence '\p'
  path = "output\pose_13.csv"


1. Load the existing data

In [22]:
data = pd.read_csv(path)
data 

,frame,time,joint,x,y,z
0,3,0.784767,0,0.530179,0.215627,-0.097599
1,3,0.784767,1,0.540962,0.206748,-0.096093
2,3,0.784767,2,0.544324,0.207425,-0.096114
3,3,0.784767,3,0.547695,0.208170,-0.096150
4,3,0.784767,4,0.532080,0.198434,-0.075081
...,...,...,...,...,...,...
19432,757,103.463765,28,0.463749,0.866676,0.367657
19433,757,103.463765,29,0.471892,0.940032,0.094553
19434,757,103.463765,30,0.472087,0.899563,0.380971
19435,757,103.463765,31,0.418385,0.930077,0.106277


1.2 Data cleanup 

Since the data was generated using MediaPipe, some joints are not relevant for my analysis. The joints are organized as follows:
| ID  | Landmark           |
|-----|------------------|
| 0   | NOSE              |
| 1   | LEFT_EYE_INNER    |
| 2   | LEFT_EYE          |
| 3   | LEFT_EYE_OUTER    |
| 4   | RIGHT_EYE_INNER   |
| 5   | RIGHT_EYE         |
| 6   | RIGHT_EYE_OUTER   |
| 7   | LEFT_EAR          |
| 8   | RIGHT_EAR         |
| 9   | MOUTH_LEFT        |
| 10  | MOUTH_RIGHT       |
| 11  | LEFT_SHOULDER     |
| 12  | RIGHT_SHOULDER    |
| 13  | LEFT_ELBOW        |
| 14  | RIGHT_ELBOW       |
| 15  | LEFT_WRIST        |
| 16  | RIGHT_WRIST       |
| 17  | LEFT_PINKY        |
| 18  | RIGHT_PINKY       |
| 19  | LEFT_INDEX        |
| 20  | RIGHT_INDEX       |
| 21  | LEFT_THUMB        |
| 22  | RIGHT_THUMB       |
| 23  | LEFT_HIP          |
| 24  | RIGHT_HIP         |
| 25  | LEFT_KNEE         |
| 26  | RIGHT_KNEE        |
| 27  | LEFT_ANKLE        |
| 28  | RIGHT_ANKLE       |
| 29  | LEFT_HEEL         |
| 30  | RIGHT_HEEL        |
| 31  | LEFT_FOOT_INDEX   |
| 32  | RIGHT_FOOT_INDEX  |


I will exclude irrelevant joints, such as ears ore pinkys, from the analysis.


In [23]:
joints = [
    (11, "LEFT_SHOULDER"),
    (12, "RIGHT_SHOULDER"),
    (13, "LEFT_ELBOW"),
    (14, "RIGHT_ELBOW"),
    (15, "LEFT_WRIST"),
    (16, "RIGHT_WRIST"),
    (23, "LEFT_HIP"),
    (24, "RIGHT_HIP"),
    (25, "LEFT_KNEE"),
    (26, "RIGHT_KNEE"),
    (27, "LEFT_ANKLE"),
    (28, "RIGHT_ANKLE"),
    (29, "LEFT_HEEL"),
    (30, "RIGHT_HEEL"),
    (31, "LEFT_FOOT_INDEX"),
    (32, "RIGHT_FOOT_INDEX")
]


In [24]:
joints_ids = [id for id, name in joints]

data_filtered = data[data["joint"].isin(joints_ids)]
data_filtered


,frame,time,joint,x,y,z
11,3,0.784767,11,0.585773,0.314607,-0.106602
12,3,0.784767,12,0.509420,0.318972,0.077063
13,3,0.784767,13,0.582659,0.288679,-0.288116
14,3,0.784767,14,0.453737,0.326861,0.038258
15,3,0.784767,15,0.541644,0.160160,-0.415120
...,...,...,...,...,...,...
19432,757,103.463765,28,0.463749,0.866676,0.367657
19433,757,103.463765,29,0.471892,0.940032,0.094553
19434,757,103.463765,30,0.472087,0.899563,0.380971
19435,757,103.463765,31,0.418385,0.930077,0.106277


1.3 Adding Unknown Data Using Linear Interpolation

In [25]:
if data_filtered.isna().any().any():
    print("there are NaN values in the data")
    data_filtered = data_filtered.interpolate(method="linear")


2. Compute joint angles


In [26]:
import numpy as np

Pivot data to have joints as columns per frame
Columns will be like: joint0_x, joint0_y, joint0_z, joint1_x, ...

In [27]:

pose_pivot = data_filtered.pivot(index='frame', columns='joint', values=['x','y','z'])
pose_pivot.columns = [f"{axis}{joint}" for axis, joint in pose_pivot.columns]


Calculate the angle at point b formed by points a-b-c in 3D.
Returns angle in degrees.


In [28]:
def calculate_angle(a, b, c):

    ba = np.array(a) - np.array(b)
    bc = np.array(c) - np.array(b)
    cos_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc))
    cos_angle = np.clip(cos_angle, -1.0, 1.0)  # Numerical stability
    angle = np.arccos(cos_angle)
    return np.degrees(angle)


2.2 Now I have to add a joint tripplets to calculat the angles between them.

In [29]:

angle_defs = {
    "elbow_left":  ("LEFT_SHOULDER", "LEFT_ELBOW", "LEFT_WRIST"),
    "elbow_right": ("RIGHT_SHOULDER", "RIGHT_ELBOW", "RIGHT_WRIST"),
    "knee_left":   ("LEFT_HIP", "LEFT_KNEE", "LEFT_ANKLE"),
    "knee_right":  ("RIGHT_HIP", "RIGHT_KNEE", "RIGHT_ANKLE"),
    "hip_left":    ("LEFT_SHOULDER", "LEFT_HIP", "LEFT_KNEE"),
    "hip_right":   ("RIGHT_SHOULDER", "RIGHT_HIP", "RIGHT_KNEE")
}

Adding the new features to the dataset

In [30]:
for angle_name in angle_defs:
    pose_pivot[angle_name] = None

In [31]:
for idx, row in pose_pivot.iterrows():
    coords = {
        name: (row[f'x{id}'], row[f'y{id}'], row[f'z{id}'])
        for id, name in joints
    }

    for angle_name, (a, b, c) in angle_defs.items():
        p1 = coords[a]
        p2 = coords[b]
        p3 = coords[c]
        pose_pivot.at[idx, angle_name] = calculate_angle(p1, p2, p3)

pose_pivot

,x11,x12,x13,x14,x15,x16,x23,x24,x25,x26,...,z29,z30,z31,z32,elbow_left,elbow_right,knee_left,knee_right,hip_left,hip_right
frame,,,,,,,,,,,,,,,,,,,,,
3,0.585773,0.509420,0.582659,0.453737,0.541644,0.438609,0.533063,0.488092,0.510907,0.439813,...,0.021261,0.130957,-0.072406,0.059207,141.289189,118.02409,70.845566,68.938423,118.209087,137.178941
4,0.588842,0.507618,0.614638,0.449715,0.580751,0.438521,0.533119,0.487137,0.503740,0.435982,...,0.021572,0.275251,-0.059492,0.227501,141.039442,130.782373,51.776467,69.495127,108.850908,142.795055
5,0.595480,0.508614,0.642400,0.449856,0.604727,0.442826,0.535058,0.489026,0.503047,0.440429,...,-0.022295,0.130965,-0.104771,0.064192,139.10368,141.914449,51.809611,67.385745,106.987083,135.907863
6,0.594183,0.511198,0.646666,0.468249,0.647200,0.454712,0.535856,0.495680,0.500758,0.445426,...,0.058928,0.138072,-0.019856,0.056628,159.457206,167.662674,82.526381,124.314067,137.30006,158.715085
7,0.581604,0.521597,0.633559,0.485925,0.652042,0.471793,0.527781,0.497952,0.488739,0.455036,...,0.432281,-0.080678,0.379077,-0.193029,161.793433,175.954516,158.19537,158.202968,143.315114,146.990172
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
753,0.556380,0.515950,0.472407,0.457359,0.411649,0.406812,0.530989,0.487233,0.499306,0.450445,...,0.117843,0.336459,-0.003334,0.283875,114.064012,138.377572,113.806727,150.677617,144.279455,149.113542
754,0.538088,0.516833,0.467121,0.455916,0.406200,0.401552,0.527927,0.497575,0.491590,0.466401,...,0.082412,0.381042,-0.002376,0.344456,111.840712,93.362893,156.089454,170.618801,164.741209,144.16429
755,0.514094,0.523563,0.443457,0.461018,0.387690,0.401292,0.517388,0.512565,0.474881,0.474767,...,0.184722,0.444569,0.131216,0.419951,124.26489,114.050839,170.36091,170.430187,165.975825,142.626177


2.3 relative distances between body parts

In [32]:
distance_defs = {
    "foot_to_foot": ("LEFT_ANKLE", "RIGHT_ANKLE"),
    "foot_to_hip": ("LEFT_ANKLE", "LEFT_HIP", "RIGHT_ANKLE", "RIGHT_HIP"),
    "hip_to_shoulder": ("LEFT_HIP", "LEFT_SHOULDER", "RIGHT_HIP", "RIGHT_SHOULDER")
}

In [33]:
def calculate_distance(a, b):
    return np.linalg.norm(np.array(a) - np.array(b))

In [34]:
def calculate_distance4(a,b,c,d):
    left = calculate_distance(a,b)
    right = calculate_distance(c,d)
    return (left + right) / 2   

In [35]:
for distance_name in distance_defs:
    pose_pivot[distance_name] = None

In [36]:
for idx, row in pose_pivot.iterrows():

    coords = {
        name: (row[f'x{id}'], row[f'y{id}'], row[f'z{id}'])
        for id, name in joints
    }

    for distance_name, joint_def in distance_defs.items():

        if len(joint_def) == 2:
            a, b = joint_def

            p1 = coords[a]
            p2 = coords[b]

            pose_pivot.at[idx, distance_name] = \
                calculate_distance(p1, p2)

        else:
            a1, b1, a2, b2 = joint_def

            p1 = coords[a1]
            p2 = coords[b1]
            p3 = coords[a2]
            p4 = coords[b2]

            pose_pivot.at[idx, distance_name] = \
                calculate_distance4(p1, p2, p3, p4)

In [37]:
pose_pivot.to_csv(path)